# 04 – Bronze Layer: NYC Motor Vehicle Collisions Ingestion

**Source:** NYC Open Data – Motor Vehicle Collisions – Crashes (`h9gi-nx95`). Same rules as the EMS Bronze notebook: raw data, fingerprint-based change detection, dated snapshots, and a run log.

## 1. Load configuration

Paths and settings are stored in `config/dev.yaml` and `config/prod.yaml`, not in the code. The environment comes from the `PIPELINE_ENV` variable (the run script sets it). If it is not set, this notebook uses `prod`.

In [11]:
import os, sys
sys.path.append("../src")          # lets the notebook import files from src/

from config_loader import load_config

ENV = os.environ.get("PIPELINE_ENV", "prod")   # "dev" = 2,000-row sample, "prod" = full data
config = load_config(f"../config/{ENV}.yaml")

print("Environment:", ENV)
print(config)

Environment: prod
{'environment': 'prod', 'spark': {'driver_memory': '4g', 'cores': 2, 'shuffle_partitions': 400}, 'source': {'dataset_name': 'ems_incident_dispatch', 'dataset_id': '76xm-jjuj', 'input_path': '/home/ubuntu/spark-notebooks/ems-pipeline/data_sample/EMS_Incident_Dispatch_Data.csv'}, 'bronze': {'root_path': '/home/ubuntu/spark-notebooks/ems-pipeline/data/bronze'}, 'silver': {'root_path': '/home/ubuntu/spark-notebooks/ems-pipeline/data/silver'}, 'gold': {'root_path': '/home/ubuntu/spark-notebooks/ems-pipeline/data/gold'}, 'collisions': {'dataset_name': 'motor_vehicle_collisions', 'dataset_id': 'h9gi-nx95', 'input_path': '/home/ubuntu/spark-notebooks/ems-pipeline/data_sample/collisions_data.csv'}}


## 2. Start Spark

Create the Spark session, which is the entry point for all Spark operations. Memory, cores and shuffle partitions come from the config, so the Bronze run uses the same settings as Silver and Gold. `getOrCreate()` reuses a running session, so restart the kernel after changing these settings.

In [12]:
from pyspark.sql import SparkSession

spark_cfg = config["spark"]

spark = (
    SparkSession.builder
    .appName("ems_bronze_ingest")
    .master(f"local[{spark_cfg['cores']}]")
    .config("spark.driver.memory", spark_cfg["driver_memory"])
    .config("spark.sql.shuffle.partitions", spark_cfg["shuffle_partitions"])
    .getOrCreate()
)

print(spark.version)
print("Master:", spark.sparkContext.master)
print("Driver memory:", spark.sparkContext.getConf().get("spark.driver.memory"))

3.5.0
Master: local[2]
Driver memory: 4g


## Performance measurement helper

Spark is lazy, so we time the actions (`count`, `write`), and each printed time covers the work behind that action.

In [13]:
import time
from contextlib import contextmanager

@contextmanager
def timed(label):
    start = time.time()
    yield
    print(f"[{label}] {time.time() - start:.1f} s")

## 3. Check whether the source file has changed

The source publishes irregularly, so a new Bronze snapshot should only be created when the data actually changed.

1. A **fingerprint** (SHA-256 checksum) of the source file is computed. Two files with the same fingerprint have identical content.
2. It is compared with the fingerprint stored in the **last successful ingestion log**.
3. **Identical:** a log entry with status `skipped_unchanged` is written and the rest of the notebook does nothing.
4. **Different, or no earlier fingerprint:** a new snapshot is ingested and its fingerprint is recorded.

If a different file arrives on the same day as an earlier snapshot, the new snapshot gets a numbered folder (`snapshot_date=YYYY-MM-DD_2`), so earlier snapshots are never overwritten.

In [14]:
import glob, hashlib, json, uuid
from datetime import datetime, timezone

run_id = str(uuid.uuid4())
snapshot_date = datetime.now(timezone.utc).strftime("%Y-%m-%d")

source_path = os.path.abspath(os.path.join("..", config["collisions"]["input_path"]))
log_dir = os.path.abspath(os.path.join("..", config["bronze"]["root_path"], "_ingestion_log"))
os.makedirs(log_dir, exist_ok=True)


def file_sha256(path, chunk_size=8 * 1024 * 1024):
    """Fingerprint of a file's content, read in chunks so the 6.5 GB file never sits in memory."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk_size), b""):
            h.update(block)
    return h.hexdigest()


def latest_successful_log():
    """The most recent ingestion log with status 'success' for this dataset, or None."""
    found = []
    for p in glob.glob(os.path.join(log_dir, f"{config['source']['dataset_name']}_*.json")):
        with open(p) as f:
            entry = json.load(f)
        if entry.get("status") == "success":
            found.append(entry)
    return max(found, key=lambda e: e["ingested_at_utc"]) if found else None


with timed("Fingerprint (SHA-256) of the source file"):
    source_sha256 = file_sha256(source_path)
source_size_bytes = os.path.getsize(source_path)

previous = latest_successful_log()
previous_sha = previous.get("source_sha256") if previous else None
source_changed = previous_sha != source_sha256

print("Source file:", source_path)
print("Fingerprint:", source_sha256)

if previous is None:
    print("No earlier ingestion found: this is the first snapshot.")
elif previous_sha is None:
    print("The earlier ingestion has no fingerprint: ingesting again and recording one.")
elif source_changed:
    print("Source changed since snapshot", previous["snapshot_date"], ": ingesting a new snapshot.")
else:
    print("Source is identical to snapshot", previous["snapshot_date"], ": nothing to ingest.")

if not source_changed:
    skip_log = {
        "status": "skipped_unchanged",
        "run_id": run_id,
        "environment": config["environment"],
        "dataset_name": config["source"]["dataset_name"],
        "dataset_id": config["source"]["dataset_id"],
        "ingested_at_utc": datetime.now(timezone.utc).isoformat(),
        "source_path": config["source"]["input_path"],
        "source_sha256": source_sha256,
        "unchanged_since_snapshot": previous["snapshot_date"],
    }
    skip_file = os.path.join(log_dir, f"{config['source']['dataset_name']}_skipped_{snapshot_date}_{run_id}.json")
    with open(skip_file, "w") as f:
        json.dump(skip_log, f, indent=2)
    print("Skip logged to:", skip_file)
    print("Bronze is finished. The run script skips Silver and Gold unless FORCE=1.")

[Fingerprint (SHA-256) of the source file] 1.6 s
Source file: /home/ubuntu/spark-notebooks/ems-pipeline/data_sample/collisions_data.csv
Fingerprint: f6fc4540d242f5c31c3b2c3c31476aa36fe5639505be5003cabbf631d0ee3ee9
Source changed since snapshot 2026-11-10 : ingesting a new snapshot.


## 4. Read the raw CSV

The file is read with `inferSchema=False`, so every column stays as text. Bronze keeps data in its original form, and type conversion happens later in the Silver layer.

**Sanity check:** the sample has 2,000 rows and the full file has 30,389,767 rows, all with 31 columns. This section does nothing if the source is unchanged.

In [15]:
if source_changed:
    raw_df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .csv(source_path)
    )

    raw_df = raw_df.toDF(*[c.strip().replace(" ", "_") for c in raw_df.columns])

    with timed("Count raw rows"):
        raw_rows = raw_df.count()

    print("Rows:", raw_rows)
    print("Columns:", len(raw_df.columns))
else:
    print("Skipped: source unchanged.")

[Count raw rows] 1.7 s
Rows: 2301383
Columns: 29


## 5. Inspect the raw data

Check the schema (all columns should be `string`) and look at a few rows to see what the raw values look like, such as dates stored as text and any empty fields.

In [17]:
if source_changed:
    raw_df.printSchema()
    raw_df.show(5, truncate=False)
else:
    print("Skipped: source unchanged.")

root
 |-- CRASH_DATE: string (nullable = true)
 |-- CRASH_TIME: string (nullable = true)
 |-- BOROUGH: string (nullable = true)
 |-- ZIP_CODE: string (nullable = true)
 |-- LATITUDE: string (nullable = true)
 |-- LONGITUDE: string (nullable = true)
 |-- LOCATION: string (nullable = true)
 |-- ON_STREET_NAME: string (nullable = true)
 |-- CROSS_STREET_NAME: string (nullable = true)
 |-- OFF_STREET_NAME: string (nullable = true)
 |-- NUMBER_OF_PERSONS_INJURED: string (nullable = true)
 |-- NUMBER_OF_PERSONS_KILLED: string (nullable = true)
 |-- NUMBER_OF_PEDESTRIANS_INJURED: string (nullable = true)
 |-- NUMBER_OF_PEDESTRIANS_KILLED: string (nullable = true)
 |-- NUMBER_OF_CYCLIST_INJURED: string (nullable = true)
 |-- NUMBER_OF_CYCLIST_KILLED: string (nullable = true)
 |-- NUMBER_OF_MOTORIST_INJURED: string (nullable = true)
 |-- NUMBER_OF_MOTORIST_KILLED: string (nullable = true)
 |-- CONTRIBUTING_FACTOR_VEHICLE_1: string (nullable = true)
 |-- CONTRIBUTING_FACTOR_VEHICLE_2: string (nu

## 6. Add traceability metadata

Bronze keeps all original columns unchanged. We add four metadata columns so every row can be traced back to its source:

- `_ingestion_timestamp`: when the data was loaded
- `_run_id`: a unique ID for this pipeline run
- `_source_file`: the file the row came from
- `_dataset_id`: the NYC Open Data dataset ID

The run ID and the snapshot date were created in section 3.

In [20]:
from pyspark.sql import functions as F

if source_changed:
    bronze_df = (
        raw_df
        .withColumn("_ingestion_timestamp", F.current_timestamp())
        .withColumn("_run_id", F.lit(run_id))
        .withColumn("_source_file", F.input_file_name())
        .withColumn("_dataset_id", F.lit(config["source"]["dataset_id"]))
    )

    print("Run ID:", run_id)
    print("Snapshot date:", snapshot_date)
    print("Columns now:", len(bronze_df.columns))

    bronze_df.select(
        "COLLISION_ID", "_ingestion_timestamp", "_run_id", "_source_file", "_dataset_id"
    ).show(3, truncate=False)
else:
    print("Skipped: source unchanged.")

Run ID: 8407f72c-1952-41cb-877d-bcccfcbc6a1a
Snapshot date: 2026-11-13
Columns now: 33
+--------------------------------+-----------------------+------------------------------------+--------------------------------------------------------------------------------+-----------+
|COLLISION_ID                    |_ingestion_timestamp   |_run_id                             |_source_file                                                                    |_dataset_id|
+--------------------------------+-----------------------+------------------------------------+--------------------------------------------------------------------------------+-----------+
|01389705cb9ad8d5f6edaa52928872a8|2026-11-13 16:02:48.531|8407f72c-1952-41cb-877d-bcccfcbc6a1a|file:///home/ubuntu/spark-notebooks/ems-pipeline/data_sample/collisions_data.csv|76xm-jjuj  |
|f3c9ecc759b4408805bc858f69dc52bc|2026-11-13 16:02:48.531|8407f72c-1952-41cb-877d-bcccfcbc6a1a|file:///home/ubuntu/spark-notebooks/ems-pipeline/data_sample/c

## 7. Write the Bronze snapshot

The data is saved as Parquet in `bronze/<dataset_name>/snapshot_date=<date>/`.

- All columns are still strings, so the data remains raw. Parquet only changes the storage format.
- This section only runs when the source changed. If a snapshot folder for today already exists, the new one gets a numbered name (`_2`, `_3`, ...). Bronze never overwrites earlier snapshots.

In [21]:
if source_changed:
    bronze_base = os.path.abspath(os.path.join(
        "..", config["bronze"]["root_path"], config["collisions"]["dataset_name"]
    ))
    snapshot_folder = f"snapshot_date={snapshot_date}"
    counter = 1
    while os.path.exists(os.path.join(bronze_base, snapshot_folder)):
        counter += 1
        snapshot_folder = f"snapshot_date={snapshot_date}_{counter}"

    bronze_path = os.path.join(bronze_base, snapshot_folder)
    print("Bronze path:", bronze_path)

    with timed("Bronze write"):
        bronze_df.write.mode("errorifexists").parquet(bronze_path)
    print("Snapshot written.")
else:
    print("Skipped: source unchanged.")

Bronze path: /home/ubuntu/spark-notebooks/ems-pipeline/data/bronze/motor_vehicle_collisions/snapshot_date=2026-11-13


[Bronze write] 15.4 s
Snapshot written.


### Verify the snapshot

Read the Parquet files back and compare them with the original input. The row count must match (the notebook stops with an error if it does not), and all columns should still be strings.

In [22]:
if source_changed:
    check_df = spark.read.parquet(bronze_path)

    with timed("Count written rows"):
        written_rows = check_df.count()

    print("Rows written:", written_rows)
    print("Rows in raw input:", raw_rows)
    assert written_rows == raw_rows, "Row counts differ between the raw input and the Bronze snapshot!"
    print("Columns:", len(check_df.columns))

    check_df.printSchema()
    print(os.listdir(bronze_path))
else:
    print("Skipped: source unchanged.")

[Count written rows] 0.2 s
Rows written: 2301383
Rows in raw input: 2301383
Columns: 33
root
 |-- CRASH_DATE: string (nullable = true)
 |-- CRASH_TIME: string (nullable = true)
 |-- BOROUGH: string (nullable = true)
 |-- ZIP_CODE: string (nullable = true)
 |-- LATITUDE: string (nullable = true)
 |-- LONGITUDE: string (nullable = true)
 |-- LOCATION: string (nullable = true)
 |-- ON_STREET_NAME: string (nullable = true)
 |-- CROSS_STREET_NAME: string (nullable = true)
 |-- OFF_STREET_NAME: string (nullable = true)
 |-- NUMBER_OF_PERSONS_INJURED: string (nullable = true)
 |-- NUMBER_OF_PERSONS_KILLED: string (nullable = true)
 |-- NUMBER_OF_PEDESTRIANS_INJURED: string (nullable = true)
 |-- NUMBER_OF_PEDESTRIANS_KILLED: string (nullable = true)
 |-- NUMBER_OF_CYCLIST_INJURED: string (nullable = true)
 |-- NUMBER_OF_CYCLIST_KILLED: string (nullable = true)
 |-- NUMBER_OF_MOTORIST_INJURED: string (nullable = true)
 |-- NUMBER_OF_MOTORIST_KILLED: string (nullable = true)
 |-- CONTRIBUTING_F

## 8. Record the run metadata

A small JSON log is written for every ingestion run. It records what was loaded, when, from where, how many rows, which columns, and the **fingerprint of the source file**, which the next run uses to decide whether anything changed. Logs are stored in `bronze/_ingestion_log/`, separate from the data snapshots.

In [25]:
if source_changed:
    run_log = {
        "run_id": run_id,
        "environment": config["environment"],
        "dataset_name": config["source"]["dataset_name"],
        "dataset_id": config["source"]["dataset_id"],
        "snapshot_date": snapshot_date,
        "snapshot_folder": os.path.basename(bronze_path),
        "ingested_at_utc": datetime.now(timezone.utc).isoformat(),
        "source_path": config["source"]["input_path"],
        "source_sha256": source_sha256,
        "source_size_bytes": source_size_bytes,
        "bronze_path": bronze_path,
        "row_count": written_rows,
        "column_count": len(check_df.columns),
        "columns": check_df.columns,
        "status": "success",
    }

    log_file = os.path.join(log_dir, f"{config['collisions']['dataset_name']}_{snapshot_date}_{run_id}.json")

    with open(log_file, "w") as f:
        json.dump(run_log, f, indent=2)

    print("Log written to:", log_file)
    print(json.dumps({k: v for k, v in run_log.items() if k != "columns"}, indent=2))
else:
    print("Skipped: source unchanged.")

Log written to: /home/ubuntu/spark-notebooks/ems-pipeline/data/bronze/_ingestion_log/motor_vehicle_collisions_2026-11-13_8407f72c-1952-41cb-877d-bcccfcbc6a1a.json
{
  "run_id": "8407f72c-1952-41cb-877d-bcccfcbc6a1a",
  "environment": "prod",
  "dataset_name": "ems_incident_dispatch",
  "dataset_id": "76xm-jjuj",
  "snapshot_date": "2026-11-13",
  "snapshot_folder": "snapshot_date=2026-11-13",
  "ingested_at_utc": "2026-11-13T16:04:49.691099+00:00",
  "source_path": "/home/ubuntu/spark-notebooks/ems-pipeline/data_sample/EMS_Incident_Dispatch_Data.csv",
  "source_sha256": "f6fc4540d242f5c31c3b2c3c31476aa36fe5639505be5003cabbf631d0ee3ee9",
  "source_size_bytes": 597727806,
  "bronze_path": "/home/ubuntu/spark-notebooks/ems-pipeline/data/bronze/motor_vehicle_collisions/snapshot_date=2026-11-13",
  "row_count": 2301383,
  "column_count": 33,
  "status": "success"
}
